# ADC Penetration Surrogate — Colab Training Pipeline

This notebook clones the repo from GitHub and runs the full pipeline end to end:

1. Install dependencies
2. (Optional) log in to Weights & Biases
3. Generate the synthetic dataset
4. Train the surrogate model
5. Evaluate the trained model and view the comparison plots

Runtime: no GPU required, but if Colab gives you one (Runtime > Change runtime type), the training cell below will use it automatically.


## 1. Clone the repository

In [ ]:
import os
REPO_URL = "https://github.com/khiemducdoan/ADC_penetration_surrogate_model.git"
REPO_DIR = "ADC_penetration_surrogate_model"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}


## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 3. (Optional) Log in to Weights & Biases

Get your API key from https://wandb.ai/authorize, then run the cell below and paste it when prompted. Skip this cell (and set `wandb.enabled=false` in the training cell) if you don't want to log this run.


In [ ]:
import wandb
wandb.login()


## 4. Configure this run

Edit these values, then the cells below use them as [Hydra](https://hydra.cc/) command-line overrides — every parameter in `configs/` can be overridden the same way.


In [ ]:
import torch

N_CONDITIONS = 2000       # number of (c0, D, r) triples to sample (100000 for a full run)
N_TIMES = 8               # time points per condition
EPOCHS = 200
HIDDEN_DIMS = "[256,256,256]"
WANDB_ENABLED = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 5. Generate the synthetic dataset

In [ ]:
!python generate_data.py \
    sampling.n_conditions={N_CONDITIONS} \
    sampling.n_times={N_TIMES}


## 6. Train the surrogate model

Logs live to Weights & Biases when `WANDB_ENABLED` is `True`.


In [ ]:
!python train.py \
    training.epochs={EPOCHS} \
    training.hidden_dims={HIDDEN_DIMS} \
    training.device={DEVICE} \
    wandb.enabled={str(WANDB_ENABLED).lower()}


## 7. Evaluate the trained model

In [ ]:
!python evaluate.py


## 8. View the comparison plots

In [ ]:
from IPython.display import Image, display

display(Image("outputs/figures/profiles_true_vs_pred.png"))
display(Image("outputs/figures/scatter_true_vs_pred.png"))


## 9. Inspect the test metrics

In [ ]:
import json

with open("outputs/test_metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))
